# ARTI406 – Assignment 2
**Dataset:** Jobs in Data  
**Tasks:** Data Quality, Missing Values, Outlier Detection, Normalization, PCA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('jobs_in_data.csv')
print('Shape:', df.shape)
df.head()

---
## Task 1 – Identify Data Quality Issues

In [ ]:
print('='*50)
print('1. Dataset Info')
print('='*50)
df.info()

In [ ]:
print('='*50)
print('2. Missing Values per Column')
print('='*50)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

plt.figure(figsize=(10, 4))
missing_pct.plot(kind='bar', color='steelblue')
plt.title('Missing Values (%) per Column')
plt.ylabel('Percentage')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print('='*50)
print('3. Duplicate Rows')
print('='*50)
dup_count = df.duplicated().sum()
print(f'Number of duplicate rows: {dup_count} ({dup_count/len(df)*100:.1f}% of dataset)')
print('\nSample duplicates:')
df[df.duplicated(keep=False)].sort_values('salary_in_usd').head(6)

In [ ]:
print('='*50)
print('4. Descriptive Statistics (Numerical)')
print('='*50)
df.describe()

In [ ]:
print('='*50)
print('5. Unique Values per Categorical Column')
print('='*50)
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f'\n{col} ({df[col].nunique()} unique values):')
    print(df[col].value_counts().head(5))

In [ ]:
print('='*50)
print('6. Data Quality Summary')
print('='*50)
print(f"""  
Issues Identified:
  • Missing values  : {df.isnull().sum().sum()} (none found in this dataset)
  • Duplicate rows  : {df.duplicated().sum()} rows ({df.duplicated().sum()/len(df)*100:.1f}%)
  • salary column   : Raw salary in local currency — not directly comparable across countries
  • salary_in_usd   : Standardised USD salary — preferred for analysis
  • work_year       : Only years 2020–2023 present (limited temporal range)
  • Potential outliers in salary_in_usd (min={df.salary_in_usd.min():,}, max={df.salary_in_usd.max():,})
""")

---
## Task 2 – Apply One Missing Value Strategy

### Rationale
The original dataset has **no missing values**. To demonstrate a realistic missing value handling strategy, we:
1. Remove duplicate rows (a genuine data quality issue).
2. Artificially introduce 5% missing values into `salary_in_usd` to simulate a real-world scenario.
3. Apply **median imputation** — the median is robust to skewness and outliers, which is important for salary data.

> **Why median over mean?** Salary distributions are typically right-skewed. The mean is pulled upward by high earners, so imputing with the mean would overestimate the typical salary. The median is a more representative central value.

In [ ]:
# Step 1: Remove duplicates
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f'Rows after removing duplicates: {len(df_clean)} (removed {len(df) - len(df_clean)} rows)')

In [ ]:
# Step 2: Introduce 5% missing values into salary_in_usd
np.random.seed(42)
df_missing = df_clean.copy()
missing_idx = np.random.choice(df_missing.index, size=int(0.05 * len(df_missing)), replace=False)
df_missing.loc[missing_idx, 'salary_in_usd'] = np.nan
print(f'Missing values introduced in salary_in_usd: {df_missing["salary_in_usd"].isnull().sum()}')

In [ ]:
# Step 3: Apply median imputation
median_salary = df_missing['salary_in_usd'].median()
print(f'Median salary_in_usd (used for imputation): ${median_salary:,.0f}')

df_imputed = df_missing.copy()
df_imputed['salary_in_usd'] = df_imputed['salary_in_usd'].fillna(median_salary)

print(f'Missing values after imputation: {df_imputed["salary_in_usd"].isnull().sum()}')

In [ ]:
# Visualise: distribution before and after imputation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_missing['salary_in_usd'].dropna(), bins=40, color='coral', edgecolor='white')
axes[0].set_title('salary_in_usd – Before Imputation\n(with missing values)')
axes[0].set_xlabel('Salary (USD)')
axes[0].set_ylabel('Count')

axes[1].hist(df_imputed['salary_in_usd'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('salary_in_usd – After Median Imputation')
axes[1].set_xlabel('Salary (USD)')

plt.tight_layout()
plt.show()

# Use df_imputed going forward
df_work = df_imputed.copy()
print('\nWorking dataset shape:', df_work.shape)

---
## Task 3 – Detect and Handle Outliers Using IQR

The IQR (Interquartile Range) method defines outliers as values below **Q1 − 1.5×IQR** or above **Q3 + 1.5×IQR**. We apply this to `salary_in_usd`, the most analytically meaningful numerical column.

In [ ]:
# Compute IQR bounds
Q1 = df_work['salary_in_usd'].quantile(0.25)
Q3 = df_work['salary_in_usd'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'Q1            : ${Q1:,.0f}')
print(f'Q3            : ${Q3:,.0f}')
print(f'IQR           : ${IQR:,.0f}')
print(f'Lower bound   : ${lower_bound:,.0f}')
print(f'Upper bound   : ${upper_bound:,.0f}')

In [ ]:
# Identify outliers
outliers = df_work[(df_work['salary_in_usd'] < lower_bound) | (df_work['salary_in_usd'] > upper_bound)]
print(f'Outliers detected: {len(outliers)} rows ({len(outliers)/len(df_work)*100:.2f}%)')
print(f'\nSalary range of outliers: ${outliers.salary_in_usd.min():,} – ${outliers.salary_in_usd.max():,}')
outliers[['job_title', 'experience_level', 'company_location', 'salary_in_usd']].head(10)

In [ ]:
# Visualise outliers
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].boxplot(df_work['salary_in_usd'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Boxplot – Before Outlier Removal')
axes[0].set_ylabel('Salary (USD)')
axes[0].set_xticklabels(['salary_in_usd'])

# Remove outliers by capping (Winsorization)
df_no_outliers = df_work.copy()
df_no_outliers['salary_in_usd'] = df_no_outliers['salary_in_usd'].clip(lower=lower_bound, upper=upper_bound)

axes[1].boxplot(df_no_outliers['salary_in_usd'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightgreen'))
axes[1].set_title('Boxplot – After Outlier Capping (IQR)')
axes[1].set_ylabel('Salary (USD)')
axes[1].set_xticklabels(['salary_in_usd'])

plt.tight_layout()
plt.show()

print(f'Before capping – max salary: ${df_work.salary_in_usd.max():,}')
print(f'After capping  – max salary: ${df_no_outliers.salary_in_usd.max():,}')

**Outlier Handling Strategy:** We used **Winsorization (capping)** — replacing values beyond the IQR bounds with the boundary values rather than dropping rows. This preserves the full dataset size while preventing extreme values from distorting analysis.

---
## Task 4 – Normalize Numerical Features (Min-Max and Z-score)

In [ ]:
# Select numerical features for normalization
num_cols = ['salary_in_usd', 'salary']
df_norm = df_no_outliers[num_cols].copy()

# --- Min-Max Normalization ---
minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(df_norm),
    columns=[c + '_minmax' for c in num_cols]
)

# --- Z-score (Standard) Normalization ---
zscore_scaler = StandardScaler()
df_zscore = pd.DataFrame(
    zscore_scaler.fit_transform(df_norm),
    columns=[c + '_zscore' for c in num_cols]
)

df_normalized = pd.concat([df_norm, df_minmax, df_zscore], axis=1)
print('Normalized dataset sample:')
df_normalized.head()

In [ ]:
print('Min-Max Normalization Statistics (should be 0–1):')
print(df_minmax.describe().round(4))

print('\nZ-score Normalization Statistics (mean≈0, std≈1):')
print(df_zscore.describe().round(4))

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 12))

for i, col in enumerate(num_cols):
    # Original
    axes[0][i].hist(df_norm[col], bins=40, color='gray', edgecolor='white')
    axes[0][i].set_title(f'{col} – Original')
    
    # Min-Max
    axes[1][i].hist(df_minmax[col + '_minmax'], bins=40, color='steelblue', edgecolor='white')
    axes[1][i].set_title(f'{col} – Min-Max Normalized (0–1)')
    
    # Z-score
    axes[2][i].hist(df_zscore[col + '_zscore'], bins=40, color='coral', edgecolor='white')
    axes[2][i].set_title(f'{col} – Z-score Normalized (mean=0)')

plt.tight_layout()
plt.show()

**Comparison:**
| Method | Formula | Range | Best for |
|---|---|---|---|
| Min-Max | (x − min) / (max − min) | [0, 1] | Algorithms sensitive to scale (KNN, Neural Nets) |
| Z-score | (x − mean) / std | Unbounded (~−3 to +3) | Normally distributed data, PCA, SVM |

---
## Task 5 – Apply PCA and Interpret Explained Variance

In [ ]:
# Encode all categorical columns with LabelEncoder
df_pca_input = df_no_outliers.copy()
le = LabelEncoder()
cat_cols = df_pca_input.select_dtypes(include='object').columns
for col in cat_cols:
    df_pca_input[col] = le.fit_transform(df_pca_input[col])

print('Columns used for PCA:')
print(df_pca_input.columns.tolist())

In [ ]:
# Z-score scale all features before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca_input)

# Apply PCA – keep all components first
pca_full = PCA()
pca_full.fit(X_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print('Explained variance per component:')
for i, (ev, cv) in enumerate(zip(explained_variance, cumulative_variance)):
    print(f'  PC{i+1}: {ev*100:.2f}%  (cumulative: {cv*100:.2f}%)')

In [ ]:
# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

n_components = len(explained_variance)
x = range(1, n_components + 1)

axes[0].bar(x, explained_variance * 100, color='steelblue', edgecolor='white')
axes[0].set_title('Scree Plot – Individual Explained Variance')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_xticks(list(x))

axes[1].plot(x, cumulative_variance * 100, marker='o', color='coral')
axes[1].axhline(y=80, color='gray', linestyle='--', label='80% threshold')
axes[1].axhline(y=95, color='black', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Principal Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_xticks(list(x))
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Determine number of components for 80% and 95% variance
n_80 = np.argmax(cumulative_variance >= 0.80) + 1
n_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f'Components needed for 80% variance: {n_80}')
print(f'Components needed for 95% variance: {n_95}')

In [ ]:
# Apply PCA with 80% threshold
pca = PCA(n_components=n_80)
X_pca = pca.fit_transform(X_scaled)

print(f'Original feature space  : {X_scaled.shape[1]} features')
print(f'Reduced PCA space       : {X_pca.shape[1]} components')
print(f'Variance retained       : {pca.explained_variance_ratio_.sum()*100:.2f}%')

# Visualise 2D projection
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
exp_labels = df_no_outliers['experience_level']
for level in exp_labels.unique():
    mask = (exp_labels == level).values
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=level, alpha=0.5, s=15)
plt.title('PCA – 2D Projection coloured by Experience Level')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(title='Experience Level', loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Feature loadings heatmap for the first 3 components
pca_3 = PCA(n_components=3)
pca_3.fit(X_scaled)
loadings = pd.DataFrame(
    pca_3.components_.T,
    index=df_pca_input.columns,
    columns=['PC1', 'PC2', 'PC3']
)

plt.figure(figsize=(8, 6))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('PCA Feature Loadings (First 3 Components)')
plt.tight_layout()
plt.show()

print('\nPC1 – top contributing features:')
print(loadings['PC1'].abs().sort_values(ascending=False).head(5))

### PCA Interpretation

- **PC1** captures the most variance and is dominated by salary-related features (`salary`, `salary_in_usd`) and location/residence — reflecting the strong link between geography and compensation.
- **PC2** is driven more by categorical features such as `employment_type`, `work_setting`, and `experience_level`, separating full-time vs. contract workers and remote vs. in-person roles.
- **PC3** contributes smaller variance and picks up finer distinctions in job category and company size.

The scree plot shows that a small number of components (≈4–6) is sufficient to retain 80%+ of the variance in the 12-feature dataset, confirming meaningful structure and significant feature redundancy.